# Notebook 6: Train Baseline Models
In this notebook, we will load our preprocessed dataset, split it into train/test sets, and train four baseline classifiers:
1. **Dummy Classifier** (naive majority class baseline)
2. **Multinomial Naive Bayes** (classic NLP classifier)
3. **Logistic Regression** (robust linear baseline)
4. **Linear Support Vector Machine (SVM)** (strong margin classifier)

We will evaluate them and save the best performing model.

In [1]:
import pandas as pd
import os
import joblib
from src.train import prepare_data, evaluate_predictions
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Load processed dataset
data_path = os.path.join("..", "data", "processed_jobs.csv")
df = pd.read_csv(data_path)

# Prepare features and split
X_train, X_test, y_train, y_test, vectorizer = prepare_data(df)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Fraud ratio in train:", y_train.mean())
print("Fraud ratio in test:", y_test.mean())

Train shape: (14304, 5003)
Test shape: (3576, 5003)
Fraud ratio in train: 0.04844798657718121
Fraud ratio in test: 0.04837807606263982


## 1. Train and Evaluate Dummy Classifier
The Dummy Classifier acts as our absolute baseline. It simply predicts the majority class (legitimate = 0) for every single sample.

In [2]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

dummy_metrics = evaluate_predictions(y_test, y_pred_dummy)
print("=== Dummy Classifier Metrics ===")
print("Accuracy:", dummy_metrics['accuracy'])
print("Precision (Fake):", dummy_metrics['precision'])
print("Recall (Fake):", dummy_metrics['recall'])
print("F1 Score:", dummy_metrics['f1_score'])
print("Confusion Matrix:\n", dummy_metrics['confusion_matrix'])

=== Dummy Classifier Metrics ===
Accuracy: 0.9516219239373602
Precision (Fake): 0.0
Recall (Fake): 0.0
F1 Score: 0.0
Confusion Matrix:
 [[3403, 0], [173, 0]]


C:\Users\alokk\OneDrive\Desktop\fake job detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\alokk\OneDrive\Desktop\fake job detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\alokk\OneDrive\Desktop\fake job detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

## 2. Train and Evaluate Naive Bayes
Naive Bayes applies Bayes' Theorem with the strong assumption that features are conditionally independent.

In [3]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
y_prob_nb = nb.predict_proba(X_test)[:, 1]

nb_metrics = evaluate_predictions(y_test, y_pred_nb, y_prob_nb)
print("=== Naive Bayes Metrics ===")
print("Accuracy:", nb_metrics['accuracy'])
print("Precision (Fake):", nb_metrics['precision'])
print("Recall (Fake):", nb_metrics['recall'])
print("F1 Score:", nb_metrics['f1_score'])
print("ROC-AUC:", nb_metrics['roc_auc'])
print("Confusion Matrix:\n", nb_metrics['confusion_matrix'])

=== Naive Bayes Metrics ===
Accuracy: 0.968400447427293
Precision (Fake): 0.8488372093023255
Recall (Fake): 0.42196531791907516
F1 Score: 0.5637065637065637
ROC-AUC: 0.9382031155780601
Confusion Matrix:
 [[3390, 13], [100, 73]]


## 3. Train and Evaluate Logistic Regression
Logistic regression maps predictions to probabilities using the logistic sigmoid function. We set `class_weight='balanced'` to offset the severe class imbalance.

In [4]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = evaluate_predictions(y_test, y_pred_lr, y_prob_lr)
print("=== Logistic Regression Metrics ===")
print("Accuracy:", lr_metrics['accuracy'])
print("Precision (Fake):", lr_metrics['precision'])
print("Recall (Fake):", lr_metrics['recall'])
print("F1 Score:", lr_metrics['f1_score'])
print("ROC-AUC:", lr_metrics['roc_auc'])
print("Confusion Matrix:\n", lr_metrics['confusion_matrix'])

=== Logistic Regression Metrics ===
Accuracy: 0.9586129753914989
Precision (Fake): 0.5432525951557093
Recall (Fake): 0.9075144508670521
F1 Score: 0.6796536796536796
ROC-AUC: 0.9892359512772647
Confusion Matrix:
 [[3271, 132], [16, 157]]


## 4. Train and Evaluate Support Vector Machine (SVM)
Support Vector Machine finds the hyperplane that maximizes the margin between classes. We use `LinearSVC` with balanced class weights.

In [5]:
svm = LinearSVC(class_weight='balanced', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# LinearSVC does not support predict_proba natively, so we pass decision_function as probability proxy for ROC-AUC
y_prob_svm = svm.decision_function(X_test)

svm_metrics = evaluate_predictions(y_test, y_pred_svm, y_prob_svm)
print("=== Linear SVM Metrics ===")
print("Accuracy:", svm_metrics['accuracy'])
print("Precision (Fake):", svm_metrics['precision'])
print("Recall (Fake):", svm_metrics['recall'])
print("F1 Score:", svm_metrics['f1_score'])
print("ROC-AUC:", svm_metrics['roc_auc'])
print("Confusion Matrix:\n", svm_metrics['confusion_matrix'])

=== Linear SVM Metrics ===
Accuracy: 0.9832214765100671
Precision (Fake): 0.8192090395480226
Recall (Fake): 0.838150289017341
F1 Score: 0.8285714285714286
ROC-AUC: 0.9864774196178482
Confusion Matrix:
 [[3371, 32], [28, 145]]


## 5. Saving the Best Model and Vectorizer
We will save our trained Logistic Regression model and the fitted TF-IDF vectorizer to the `models/` directory for production use.

In [6]:
# Ensure models directory exists
os.makedirs(os.path.join("..", "models"), exist_ok=True)

model_path = os.path.join("..", "models", "logistic_regression.pkl")
vectorizer_path = os.path.join("..", "models", "tfidf_vectorizer.pkl")

joblib.dump(lr, model_path)
joblib.dump(vectorizer, vectorizer_path)

print("Successfully saved trained model to:", model_path)
print("Successfully saved TF-IDF Vectorizer to:", vectorizer_path)

Successfully saved trained model to: ..\models\logistic_regression.pkl
Successfully saved TF-IDF Vectorizer to: ..\models\tfidf_vectorizer.pkl
